In [23]:
!pip install transformers -q

In [24]:
import torch
import numpy as np
import random
import re
import os
import pandas as pd
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [25]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("🔥 Using device:", device)

🔥 Using device: cuda


In [26]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [27]:
def preprocess_text(text):
    """
    Preprocess tweet text:
    - Remove URLs
    - Remove @mentions
    - Keep hashtags
    - Lowercase
    """
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    # Remove @mentions
    text = re.sub(r'@\w+', '', text)
    # Lowercase
    text = text.lower()
    # Remove extra whitespace
    text = ' '.join(text.split())
    return text

In [28]:
def load_twitter16_dataset(tweets_path, labels_path):
    """
    Load Twitter16 dataset from source_tweets.txt and label.txt
    Returns: DataFrame with columns [text, label, event]
    """
    # Load tweets
    tweets = {}
    with open(tweets_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t', 1)
            if len(parts) == 2:
                tweet_id, text = parts
                tweets[tweet_id] = text
    
    # Load labels
    data = []
    with open(labels_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split(':')
            if len(parts) == 2:
                label_str, tweet_id = parts
                if tweet_id in tweets:
                    # Map labels: true/false -> 1/0 (rumor/non-rumor)
                    if label_str in ['true', 'unverified']:
                        label = 1
                    else:  # 'false', 'non-rumor'
                        label = 0
                    
                    text = preprocess_text(tweets[tweet_id])
                    data.append({'text': text, 'label': label, 'event': 'twitter16'})
    
    return pd.DataFrame(data)

print("✅ Twitter16 loading function defined!")

# Load Twitter16 dataset
twitter16_df = load_twitter16_dataset(
    '/kaggle/input/datasets/nsm177/twitter16/source_tweets.txt',
    '/kaggle/input/datasets/nsm177/twitter16/label.txt'
)

print(f"\n✅ Twitter16 dataset: {len(twitter16_df):,} samples")

# Show sample data
print("\n📊 Sample from Twitter16 dataset:")
print(twitter16_df.head(3))


✅ Twitter16 loading function defined!

✅ Twitter16 dataset: 818 samples

📊 Sample from Twitter16 dataset:
                                                text  label      event
0   correct predictions in back to the future ii url      0  twitter16
1  . in rainbow colors for #scotusmarriage? here'...      1  twitter16
2  cops bought the alleged church shooter burger ...      0  twitter16


In [29]:
print("="*60)
print("🔀 Performing TEMPORAL Split (Twitter16 Only)")
print("="*60)

# Temporal split for Twitter16
# Use first 80% as training, last 20% as emergent test
twitter16_split = int(len(twitter16_df) * 0.8)
train_df = twitter16_df.iloc[:twitter16_split]
test_df = twitter16_df.iloc[twitter16_split:]

print(f"\n📊 Twitter16 temporal split (80/20):")
print(f"   Training samples: {len(train_df):,}")
print(f"   Test samples (emergent): {len(test_df):,}")

print(f"\n📊 Label distribution:")
print(f"   Train: {dict(train_df['label'].value_counts())}")
print(f"   Test:  {dict(test_df['label'].value_counts())}")


🔀 Performing TEMPORAL Split (Twitter16 Only)

📊 Twitter16 temporal split (80/20):
   Training samples: 654
   Test samples (emergent): 164

📊 Label distribution:
   Train: {0: np.int64(358), 1: np.int64(296)}
   Test:  {1: np.int64(112), 0: np.int64(52)}


## 8. Load and Prepare Twitter16 Dataset

In [30]:
class FakeNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

## 9. Perform Temporal Data Splitting (Twitter16 Only)

In [31]:
print("="*60)
print("🔧 Initializing Tokenizer and DataLoaders")
print("="*60)

# Hyperparameters
MAX_LENGTH = 256
BATCH_SIZE = 32

# Initialize tokenizer
print("\n📝 Loading RoBERTa tokenizer...")
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# Create datasets
train_dataset = FakeNewsDataset(
    train_df['text'].tolist(),
    train_df['label'].tolist(),
    tokenizer,
    MAX_LENGTH
)
test_dataset = FakeNewsDataset(
    test_df['text'].tolist(),
    test_df['label'].tolist(),
    tokenizer,
    MAX_LENGTH
)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print(f"✅ Train batches: {len(train_loader)}")
print(f"✅ Test batches: {len(test_loader)}")
print(f"✅ Batch size: {BATCH_SIZE}")

🔧 Initializing Tokenizer and DataLoaders

📝 Loading RoBERTa tokenizer...
✅ Train batches: 21
✅ Test batches: 6
✅ Batch size: 32


## 10. Initialize Tokenizer and DataLoaders

In [32]:
print("="*60)
print("🤖 Initializing RoBERTa Model")
print("="*60)

# Hyperparameters
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4
EPOCHS = 5 # Reduced for testing

# Initialize model
print("\n📥 Loading roberta-base model...")
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

# Move model to GPU
model.to(device)
print(f"✅ Model moved to {device}")

# Initialize optimizer
optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Scheduler
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

print(f"✅ Optimizer: AdamW (lr={LEARNING_RATE}, weight_decay={WEIGHT_DECAY})")
print(f"✅ Scheduler: Linear warmup (total_steps={total_steps})")
print(f"✅ Model parameters: {sum(p.numel() for p in model.parameters()):,}")


🤖 Initializing RoBERTa Model

📥 Loading roberta-base model...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model moved to cuda
✅ Optimizer: AdamW (lr=1e-05, weight_decay=0.0001)
✅ Scheduler: Linear warmup (total_steps=105)
✅ Model parameters: 124,647,170


## 11. Initialize Model and Optimizer

## 12. Execute Training Loop with Real-time Progress

In [33]:
def train_epoch():
    model.train()
    total_loss = 0

    progress_bar = tqdm(train_loader, desc="Training", leave=True)

    for batch in progress_bar:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        # Update progress bar
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})

    return total_loss / len(train_loader)

## 13. Display Final Evaluation Metrics

In [34]:
def evaluate():
    model.eval()
    preds = []
    true_labels = []

    progress_bar = tqdm(test_loader, desc="Evaluating", leave=True)

    with torch.no_grad():
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            predictions = torch.argmax(outputs.logits, dim=1)

            preds.extend(predictions.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(true_labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, preds, average='binary', zero_division=0
    )

    return acc, precision, recall, f1

## 14. Save Trained Model

In [35]:
print("="*60)
print("🚀 STARTING TRAINING (Twitter16 Only)")
print("="*60)

for epoch in range(EPOCHS):
    print(f"\n🔥 Epoch {epoch+1}/{EPOCHS}")
    
    train_loss = train_epoch()
    acc, precision, recall, f1 = evaluate()

    print(f"\n📊 Results for Epoch {epoch+1}:")
    print(f"   Loss: {train_loss:.4f}")
    print(f"   Accuracy: {acc:.4f}")
    print(f"   Precision: {precision:.4f}")
    print(f"   Recall: {recall:.4f}")
    print(f"   F1-score: {f1:.4f}")
    print("-" * 40)


🚀 STARTING TRAINING (Twitter16 Only)

🔥 Epoch 1/5


Training:   0%|          | 0/21 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]


📊 Results for Epoch 1:
   Loss: 0.6897
   Accuracy: 0.3171
   Precision: 0.0000
   Recall: 0.0000
   F1-score: 0.0000
----------------------------------------

🔥 Epoch 2/5


Training:   0%|          | 0/21 [00:00<?, ?it/s]

...

Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]


📊 Results for Epoch 4:
   Loss: 0.5101
   Accuracy: 0.7927
   Precision: 0.9149
   Recall: 0.7679
   F1-score: 0.8350
----------------------------------------

🔥 Epoch 5/5


Training:   0%|          | 0/21 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]


📊 Results for Epoch 5:
   Loss: 0.4520
   Accuracy: 0.7866
   Precision: 0.9140
   Recall: 0.7589
   F1-score: 0.8293
----------------------------------------


In [36]:
save_path = "model_SLM_twitter16"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Model saved to", save_path)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to model_SLM_twitter16


In [37]:
print("="*60)
print("📥 Loading Saved Model for Evaluation")
print("="*60)

# Load saved model and tokenizer
loaded_model = RobertaForSequenceClassification.from_pretrained(save_path)
loaded_tokenizer = RobertaTokenizer.from_pretrained(save_path)

# Move model to device
loaded_model.to(device)
loaded_model.eval()

print(f"✅ Model loaded from {save_path}")
print(f"✅ Model moved to {device}")


📥 Loading Saved Model for Evaluation


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Model loaded from model_SLM_twitter16
✅ Model moved to cuda


In [38]:
def evaluate_loaded(model, dataloader, device):
    """Evaluate loaded model on test set"""
    model.eval()
    preds = []
    true_labels = []

    progress_bar = tqdm(dataloader, desc="Evaluating Loaded Model", leave=True)

    with torch.no_grad():
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            predictions = torch.argmax(outputs.logits, dim=1)

            preds.extend(predictions.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(true_labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, preds, average='binary', zero_division=0
    )

    return acc, precision, recall, f1

# Evaluate loaded model
print("\n🔍 Evaluating Loaded Model on Test Set...")
acc, precision, recall, f1 = evaluate_loaded(loaded_model, test_loader, device)

print(f"\n📊 Final Evaluation Results:")
print(f"   Accuracy: {acc:.4f}")
print(f"   Precision: {precision:.4f}")
print(f"   Recall: {recall:.4f}")
print(f"   F1-score: {f1:.4f}")
print("="*60)



🔍 Evaluating Loaded Model on Test Set...


Evaluating Loaded Model:   0%|          | 0/6 [00:00<?, ?it/s]


📊 Final Evaluation Results:
   Accuracy: 0.7866
   Precision: 0.9140
   Recall: 0.7589
   F1-score: 0.8293
